In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Read debug logs
data = np.loadtxt('debug_log.txt')

In [ ]:
data.shape

In [ ]:
data_after_one_step,data_initial = data[data[:,1]>0.01,:],data[data[:,1]<0.011,:]

In [ ]:
inter = data_after_one_step[np.where(data_after_one_step[:,0]<0.5)[0]]
boun = data_after_one_step[np.where(data_after_one_step[:,0]>0.5)[0]]

int_initial = data_initial[np.where(data_initial[:,0]<0.5)[0]]
boun_initial = data_initial[np.where(data_initial[:,0]>0.5)[0]]

In [ ]:
interior ={
    't' : inter[:,1],
    'x' : inter[:,2],
    'y' : inter[:,3],
    'h' : inter[:,4],
    'hu' : inter[:,5],
    'hv' : inter[:,6],
    'bath': inter[:,7],
    'capa': inter[:,8],
    'lengthr': inter[:,9],
    'indicator': inter[:,10]
}
boundary ={
    't' : boun[:,1],
    'x' : boun[:,2],
    'y' : boun[:,3],
    'h' : boun[:,4],
    'hu' : boun[:,5],
    'hv' : boun[:,6],
    'bath': boun[:,7],
    'capa': boun[:,8],
    'lengthr': boun[:,9],
    'indicator': boun[:,10]
}

interior_initial ={
    't' : int_initial[:,1],
    'x' : int_initial[:,2],
    'y' : int_initial[:,3],
    'h' : int_initial[:,4],
    'hu' : int_initial[:,5],
    'hv' : int_initial[:,6],
    'bath': int_initial[:,7],
    'capa': int_initial[:,8],
    'lengthr': int_initial[:,9],
    'indicator': int_initial[:,10]
}
boundary_initial ={
    't' : boun_initial[:,1],
    'x' : boun_initial[:,2],
    'y' : boun_initial[:,3],
    'h' : boun_initial[:,4],
    'hu' : boun_initial[:,5],
    'hv' : boun_initial[:,6],
    'bath': boun_initial[:,7],
    'capa': boun_initial[:,8],
    'lengthr': boun_initial[:,9],
    'indicator': boun_initial[:,10]
}

#### "Indicator" will be the additional dummy auax variable, which we only define for interior cells in setaux.f90 (loop from 1 to mx, my). Below one can see that it is the only aux variable for which the boundary and interior values differ.

In [ ]:
variable = "h" #["h","hu","hv","bath","capa","lengthr","indicator"]
#Color map of each
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
sc1 = ax1.scatter(interior_initial["x"], interior_initial["y"], c=interior_initial[variable], edgecolors="black",cmap='viridis', s=40,marker='s')
ax1.set_title('Interior-boundary Cells')
ax1.set_aspect('equal', 'box')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
#Plot a red rectangle from (-92,0) to (-90,-2), just the boundary_initial of it
rect = plt.Rectangle((-92, -2), 2, 2, linewidth=2, edgecolor='red', facecolor='none')
ax1.add_patch(rect)
#Plot a red line from (-92,-1) to (-90,-1), just the boundary_initial of it
hline = plt.Line2D([-92, -90], [-1, -1], linewidth=2, color='red')
ax1.add_line(hline)
vline = plt.Line2D([-91, -91], [-2, 0], linewidth=2, color='red')
ax1.add_line(vline)
plt.colorbar(sc1, ax=ax1, label='Value')    
sc2 = ax2.scatter(boundary_initial["x"], boundary_initial["y"], c=boundary_initial[variable], edgecolors="black",cmap='viridis', s=40,marker='s')
ax2.set_title('Ghost Cells')
ax2.set_xlabel('x')
#Fixed color limits for better comparison
vmin = np.min(interior_initial[variable])
vmax = np.max(interior_initial[variable])
sc1.set_clim(vmin, vmax)
vmin =  np.min(boundary_initial[variable])
vmax =  np.max(boundary_initial[variable])
sc2.set_clim(vmin, vmax)
ax2.set_aspect('equal', 'box')
rect = plt.Rectangle((-92, -2), 2, 2, linewidth=2, edgecolor='red', facecolor='none')
#Plot a red line from (-92,-1) to (-90,-1), just the boundary_initial of it
hline = plt.Line2D([-92, -90], [-1, -1], linewidth=2, color='red')
ax2.add_line(hline)
vline = plt.Line2D([-91, -91], [-2, 0], linewidth=2, color='red')
ax2.add_line(vline)
# plt.colorbar(sc1, ax=ax1, label='Value')  
ax2.add_patch(rect)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
plt.colorbar(sc2, ax=ax2, label='Value')    
plt.tight_layout()
fig.suptitle(f"{variable} at initial time", y=1.05, fontsize=16)


### Get points that coincide in space for diffs

In [ ]:
#At initial time
matching_pairs_initial = []
for i, int_row in enumerate(int_initial):
    for j, bnd_row in enumerate(boun_initial):
        if np.allclose(int_row[2:4],bnd_row[2:4]):
            matching_pairs_initial.append((i, j))
matching_pairs_initial = np.array(matching_pairs_initial)


#After one time step
matching_pairs = []
for i, int_row in enumerate(inter):
    for j, bnd_row in enumerate(boun):
        if np.allclose(int_row[2:4],bnd_row[2:4]):
            matching_pairs.append((i, j))

matching_pairs = np.array(matching_pairs)

In [ ]:
#Color map of each
variable = "indicator"
fig, ax = plt.subplots(figsize=(12, 6))
tup = (matching_pairs[:,0], matching_pairs[:,1])
# diff = (interior[matching_pairs[:,0]]-boundary[matching_pairs[:,1]])
diff = interior[variable][tup[0]]-boundary[variable][tup[1]]
sc = ax.scatter(interior["x"][tup[0]], interior["y"][tup[0]], c=diff, edgecolors="black",cmap='viridis', s=40,marker='s')
ax.set_title(f'Diff in {variable} (interior - boundary) after one time step')
#Plot a red rectangle from (-92,0) to (-90,-2), just the boundary of it
rect = plt.Rectangle((-92, -2), 2, 2, linewidth=2, edgecolor='red', facecolor='none')
ax.add_patch(rect)
#Plot a red line from (-92,-1) to (-90,-1), just the boundary of it
hline = plt.Line2D([-92, -90], [-1, -1], linewidth=2, color='red')
ax.add_line(hline)
vline = plt.Line2D([-91, -91], [-2, 0], linewidth=2, color='red')
ax.add_line(vline)
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.colorbar(sc, ax=ax, label='Value')    
# sc.set_clim(-1.e-10, 1.e-10)
ax.set_aspect('equal', 'box')
plt.tight_layout()
plt.show()